##### Understanding Fashion-MNIST
Fashion-MNIST is a dataset containing images of clothing items.

It has:

- 60,000 training images.

- 10,000 testing images.

- 10 clothing categories.

- Image dimensions of 28 × 28 pixels.

- Grayscale images.

The ten categories are:

| Label | Clothing item |
| --- | --- |
| 0 | T-shirt/top |
| 1 | Trouser |
| 2 | Pullover |
| 3 | Dress |
| 4 | Coat |
| 5 | Sandal |
| 6 | Shirt |
| 7 | Sneaker |
| 8 | Bag |
| 9 | Ankle boot |

## Import libraries and load the dataset

In [1]:
import tensorflow as tf
from tensorflow import keras

'''
The dataset is automatically downloaded 
if it is not already available locally.
'''


(x_train, y_train), (x_test, y_test) = (
    keras.datasets.fashion_mnist.load_data()
)

## Add the channel dimension 

In [2]:
x_train = x_train[..., None]

x_test = x_test[..., None]

Originally, the images have the shape:
$$
(28,28)
$$

After adding the channel dimension:
$$
(28,28,1)
$$

Why?

CNNs generally expect an explicit channel dimension.

- The value 1 indicates grayscale.

## Step 3: Load pretrained MobileNetV2
MobileNetV2 is a lightweight CNN designed to achieve efficient image processing with relatively low computational requirements.

In [3]:
base = keras.applications.MobileNetV2(
    input_shape=(96, 96, 3),
    alpha=0.35,
    include_top=False,
    weights='imagenet'
)

2019640/2019640 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Understanding the parameters:

- `input_shape=(96,96,3)`: Expects RGB images of size 96 × 96.

- `alpha=0.35`: Uses a narrower model configuration.

- `include_top=False`: Removes the original classification head.

- `weights='imagenet'`: Loads pretrained weights.

## Step 4: Freeze the pretrained model

In [4]:
base.trainable = False

This is extremely important.

- It means the pretrained MobileNetV2 layers will not be updated during training.

- The model will reuse the existing features and learn a new classification head.

## Step 5: Build the complete model

In [5]:
model = keras.Sequential([

    keras.Input(shape=(28, 28, 1)),

    keras.layers.Resizing(96, 96),

    keras.layers.Lambda(
        lambda image: tf.image.grayscale_to_rgb(image)
    ),

    keras.layers.Rescaling(
        1 / 127.5,
        offset=-1
    ),

    base,

    keras.layers.GlobalAveragePooling2D(),

    keras.layers.Dense(
        10,
        activation='softmax'
    )
])

<img src="img/understanding_1.png" width='500'>
<img src="img/understanding_2.png" width='500'>
<img src="img/understanding_3.png" width='500'>
<img src="img/understanding_4.png" width='500'>



## Step 6: Compile the model

In [6]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resizing (Resizing)             │ (None, 96, 96, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_0.35_96             │ (None, 3, 3, 1280)     │       410,208 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 423,018 (1.61 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 410,208 (1.56 MB)

**Understanding the components:**

| Component | Purpose |
| --- | --- |
| Adam | Optimisation algorithm |
| Sparse categorical crossentropy | Loss function for integer class labels |
| Accuracy | Measures correct predictions |

## Step 7: Train the model

In [7]:
history = model.fit(
    x_train,
    y_train,
    epochs=3,
    validation_split=0.2
)

Epoch 1/3
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 506s 329ms/step - accuracy: 0.8583 - loss: 0.3997 - val_accuracy: 0.8806 - val_loss: 0.3280
Epoch 2/3
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 450s 300ms/step - accuracy: 0.8930 - loss: 0.2983 - val_accuracy: 0.8918 - val_loss: 0.2954
Epoch 3/3
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 421s 280ms/step - accuracy: 0.9002 - loss: 0.2743 - val_accuracy: 0.8856 - val_loss: 0.3098


What does this mean?
- `x_train`: Training images.

- `y_train`: Correct clothing labels.

- `epochs=3`: The model goes through the training dataset three times.

- `validation_split=0.2`: 20% of the training data is reserved for validation.

## Step 8: Evaluate the model

In [8]:
test_loss, test_accuracy = model.evaluate(
    x_test,
    y_test
)

print("Test loss:", test_loss)

print("Test accuracy:", test_accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 125s 378ms/step - accuracy: 0.8853 - loss: 0.3217
Test loss: 0.32171204686164856
Test accuracy: 0.8852999806404114


## Step 9: Fine-tuning

Now we allow some pretrained layers to learn.

In [ ]:
base.trainable = True

for layer in base.layers[:-20]:
    layer.trainable = False

What happens?
- Most earlier layers remain frozen.
- The final 20 layers become trainable.
The notebook also keeps batch normalisation layers frozen:

In [ ]:
for layer in base.layers:

    if isinstance(
        layer,
        keras.layers.BatchNormalization
    ):
        layer.trainable = False

Why?

Keeping batch normalisation layers frozen can help preserve their pretrained statistics when fine-tuning on a smaller dataset.

Recompile the model:

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-5),

    loss='sparse_categorical_crossentropy',

    metrics=['accuracy']
)

Why use 1e-5?
$$
1e^{-5}=0.00001
$$

A small learning rate helps avoid making large changes to the pretrained weights.

Train again:

In [ ]:
history_finetune = model.fit(
    x_train,
    y_train,
    epochs=2,
    validation_split=0.2
)

In [ ]:
### Evaluate 
model.evaluate(x_test, y_test)